<a href="https://colab.research.google.com/github/wandb/examples/blob/master/colabs/pytorch/Simple_PyTorch_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<!--- @wandbcode{pytorch-video} -->

<img src="http://wandb.me/logo-im-png" width="400" alt="Weights & Biases" />

<!--- @wandbcode{pytorch-video} -->


# W&B + PyTorch

This notebook and its accompanying [tutorial](https://docs.wandb.ai/models/tutorials/pytorch) demonstrate how to add [Weights & Biases (W&B)](https://wandb.ai/) experiment tracking to a PyTorch training pipeline.

Use W&B for machine learning experiment tracking, dataset versioning, and project collaboration.

<div><img /></div>

<img src="https://wandb.me/mini-diagram" width="650" alt="W&B platform features: Experiments, Reports, Artifacts, Tables, Sweeps, Launch, and Models" />

<div><img /></div>




## What this notebook covers

This notebook demonstrates how to add W&B experiment tracking to a PyTorch training pipeline. It walks through a simple MNIST classifier, showing how to log hyperparameters, track model gradients, and visualize training metrics in an interactive W&B dashboard.

## The resulting interactive W&B dashboard will look like:
![W&B Charts tab showing loss and gradient metrics during training](https://i.imgur.com/z8TK2Et.png)

## In pseudocode, here's what this notebook does:

```python
import wandb

config = {"learning_rate": 0.001, "epochs": 100, "batch_size": 128}

with wandb.init(project="new-sota-model", config=config):
    # Use wandb.config for consistent logging
    config = wandb.config
    model, dataloader = get_model(), get_data()

    # Track gradients (optional)
    wandb.watch(model)

    for batch in dataloader:
        metrics = model.training_step()
        # Log metrics to W&B
        wandb.log(metrics)

    # Save model to W&B (optional)
    wandb.save("model.onnx")
```


## Follow along with a [video tutorial](http://wandb.me/pytorch-video)!
**Tip:** Already have a training pipeline? The sections marked **Step** show exactly what to add for W&B integration. Skip the data loading and model definition code.

# Install, import, and log in

### Step 1: Install W&B  and PyTorch prerequisites

Install `wandb` and the required ONNX packages. The following code also imports the required libraries and configures PyTorch for reproducibility.

In [ ]:
!pip install wandb onnx onnxscript -Uq

In [ ]:
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(hash("improves reproducibility") % 2**32 - 1)
torch.manual_seed(hash("by removing stochasticity") % 2**32 - 1)
torch.cuda.manual_seed_all(hash("so runs are repeatable") % 2**32 - 1)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Remove slow mirror from list of MNIST mirrors
torchvision.datasets.MNIST.mirrors = [mirror for mirror in torchvision.datasets.MNIST.mirrors
                                      if not mirror.startswith("http://yann.lecun.com")]

### Step 2: Import and log in to W&B

Import and log in to W&B. If this is your first time using W&B, you'll be prompted to [sign up for a free account](https://wandb.ai/authorize) or use an existing one.

When prompted, paste your API key from [wandb.ai/authorize](https://wandb.ai/authorize) and press **Enter**. The notebook saves the key for future use.

In [ ]:
import wandb

In [ ]:
wandb.login()

# Define the experiment and pipeline

## Step 3: Track metadata and hyperparameters with `wandb.init()`

Define a `config` dictionary with your hyperparameters and metadata. This example shows a few hyperparameters, but you can include any model configuration. The metadata fields (`dataset`, `architecture`) help filter and compare runs in your dashboard.

In [ ]:
config = dict(
    epochs=5,
    classes=10,
    kernels=[16, 32],
    batch_size=128,
    learning_rate=0.005,
    dataset="MNIST",
    architecture="CNN")

Wrap your training pipeline in `wandb.init()` to log hyperparameters automatically. Use `wandb.config` throughout your code to ensure the logged values match the ones used in execution.

In [ ]:
def model_pipeline(hyperparameters):

    with wandb.init(project="pytorch-demo", config=hyperparameters):
      # Access all HPs through wandb.config, so logging matches execution
      config = wandb.config

      # Make the model, data, and optimization problem
      model, train_loader, test_loader, criterion, optimizer = make(config)
      print(model)

      # Train the model
      train(model, train_loader, criterion, optimizer, config)

      # Test its final performance
      test(model, test_loader)

    return model

Calling `wandb.init` sets up communication between your code and W&B servers. Passing the `config` dictionary immediately logs your hyperparameters.

To ensure the values you log match the values used in your model, access hyperparameters through `wandb.config`. See the `make` function below for examples.

**Note:** W&B code runs in separate processes, so logging issues won't crash your training run. If the initial logging didn't work, you can log the data later with `wandb sync`.

In [ ]:
def make(config):
    # Make the data
    train, test = get_data(train=True), get_data(train=False)
    train_loader = make_loader(train, batch_size=config.batch_size)
    test_loader = make_loader(test, batch_size=config.batch_size)

    # Make the model
    model = ConvNet(config.kernels, config.classes).to(device)

    # Make the loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=config.learning_rate)

    return model, train_loader, test_loader, criterion, optimizer

# Define the data loading and model

The following is standard PyTorch data loading with no W&B-specific code.

In [ ]:
def get_data(slice=5, train=True):
    full_dataset = torchvision.datasets.MNIST(root=".",
                                              train=train,
                                              transform=transforms.ToTensor(),
                                              download=True)
    #  equiv to slicing with [::slice]
    sub_dataset = torch.utils.data.Subset(
      full_dataset, indices=range(0, len(full_dataset), slice))

    return sub_dataset


def make_loader(dataset, batch_size):
    loader = torch.utils.data.DataLoader(dataset=dataset,
                                         batch_size=batch_size,
                                         shuffle=True,
                                         pin_memory=True, num_workers=2)
    return loader

The model definition is standard PyTorch. You can experiment with the architecture, and W&B will track all your results.

In [ ]:
class ConvNet(nn.Module):
    def __init__(self, kernels, classes=10):
        super(ConvNet, self).__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(1, kernels[0], kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(kernels[0], kernels[1], kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.fc = nn.Linear(7 * 7 * kernels[-1], classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        return out

# Define training logic

Next, define the training functions. Two W&B functions come into play here: `watch` and `log`.

### Step 4: Track gradients with `wandb.watch()` and metrics with `wandb.log()`

Call `wandb.watch()` before training to log gradients and model parameters at regular intervals, as specified by `log_freq`. The rest is standard PyTorch training.

In [ ]:
def train(model, loader, criterion, optimizer, config):
    # Track gradients and weights
    wandb.watch(model, criterion, log="all", log_freq=10)

    # Run training and track with wandb
    total_batches = len(loader) * config.epochs
    example_ct = 0  # Number of examples seen
    batch_ct = 0
    for epoch in tqdm(range(config.epochs)):
        for _, (images, labels) in enumerate(loader):

            loss = train_batch(images, labels, model, optimizer, criterion)
            example_ct +=  len(images)
            batch_ct += 1

            # Report metrics every 25th batch
            if ((batch_ct + 1) % 25) == 0:
                train_log(loss, example_ct, epoch)


def train_batch(images, labels, model, optimizer, criterion):
    images, labels = images.to(device), labels.to(device)

    # Forward pass ➡
    outputs = model(images)
    loss = criterion(outputs, labels)

    # Backward pass ⬅
    optimizer.zero_grad()
    loss.backward()

    # Step with optimizer
    optimizer.step()

    return loss

Use `wandb.log()` to record metrics. Pass a dictionary with strings as keys and an optional `step` parameter.

**Tip:** This example uses the example count for the step, which makes comparison across batch sizes easier. You can also use batch count or epoch.

In [ ]:
def train_log(loss, example_ct, epoch):
    # Log metrics to W&B
    wandb.log({"epoch": epoch, "loss": loss}, step=example_ct)
    print(f"Loss after {str(example_ct).zfill(5)} examples: {loss:.3f}")

# Define testing logic

After training completes, test the model's performance.

#### (Optional) Step 5: Save the model with `wandb.save`

Export the model in [ONNX format](https://onnx.ai/) and call `wandb.save()` to upload it to W&B. This keeps model files tied to their training runs.

For more advanced features for storing, versioning, and distributing models, see [Artifacts](https://www.wandb.com/artifacts).

In [ ]:
def test(model, test_loader):
    model.eval()

    with torch.no_grad():
        correct, total = 0, 0
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f"Accuracy of the model on the {total} " +
              f"test images: {correct / total:%}")

        # Log test accuracy to W&B
        wandb.log({"test_accuracy": correct / total})

    # Save model to W&B
    torch.onnx.export(model, images, "model.onnx")
    wandb.save("model.onnx")

# Run training

Run the pipeline to start your tracked experiment. W&B outputs links to your run and project pages.

- **Run page:** Results for this specific run.
- **Project page:** All runs in your project.

On the run page, explore:

- **Charts:** Model gradients, parameter values, and loss over training.
- **System:** System metrics including disk I/O, CPU, and GPU utilization.
- **Logs:** `stdout` from training.
- **Files:** Select `model.onnx` to view your network in the [Netron model viewer](https://github.com/lutzroeder/netron).

When training completes, W&B prints a summary in the cell output.

In [ ]:
# Build, train and analyze the model with the pipeline
model = model_pipeline(config)

# Next steps

- [Sweeps](https://docs.wandb.ai/models/sweeps): This example used a single set of hyperparameters. To automate hyperparameter testing, use W&B Sweeps. See the [Hyperparameter Sweeps in PyTorch notebook](https://colab.research.google.com/github/wandb/examples/blob/master/colabs/pytorch/Organizing_Hyperparameter_Sweeps_in_PyTorch_with_W%26B.ipynb) for a full walkthrough.
- [Environment variables](https://docs.wandb.ai/platform/hosting/env-vars): Set API keys for managed clusters.
- [Offline mode](https://docs.wandb.ai/models/support/run_wandb_offline): Train offline and sync results later.
- [On-prem deployment](https://docs.wandb.ai/platform/hosting/hosting-options/self-managed): Install W&B on private or air-gapped infrastructure.
- [Customers & case studies](https://wandb.ai/site/customers/): Browse customer stories for example projects.